<a href="https://colab.research.google.com/github/Hackathon-D-Group-2026/Hackathon_D_Group_2026/blob/main/NagaVinodh_Final_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
model_zoo = {
    "Logistic_L2": Pipeline([
        ("p", Pipe_Linear_Flag),
        ("m", LogisticRegression(C=0.20, max_iter=4000, random_state=RNG_SEED)),
    ]),
    "MLP_NeuralNet": Pipeline([
        ("p", Pipe_Linear_Flag),
        ("m", MLPClassifier(
            hidden_layer_sizes=(48, 24), alpha=0.08, learning_rate_init=1e-3,
            early_stopping=True, n_iter_no_change=12, max_iter=400, random_state=RNG_SEED
        )),
    ]),
    "RandomForest": Pipeline([
        ("p", Pipe_Tree_Sentinel),
        ("m", RandomForestClassifier(
            n_estimators=450, max_depth=14, min_samples_leaf=7,
            max_features="sqrt", n_jobs=1, random_state=RNG_SEED
        )),
    ]),
    "ExtraTrees": Pipeline([
        ("p", Pipe_Tree_Sentinel),
        ("m", ExtraTreesClassifier(
            n_estimators=450, max_depth=16, min_samples_leaf=7,
            max_features=0.45, n_jobs=1, random_state=RNG_SEED
        )),
    ]),
    "HistGradBoost": Pipeline([
        ("p", Pipe_Tree_Native),
        ("m", HistGradientBoostingClassifier(
            learning_rate=0.04, max_iter=280, max_leaf_nodes=23,
            min_samples_leaf=35, l2_regularization=2.0,
            categorical_features=CAT_POS, random_state=RNG_SEED
        )),
    ]),
    "LightGBM": Pipeline([
        ("p", Pipe_Tree_Native),
        ("m", LGBMClassifier(
            n_estimators=380, learning_rate=0.03, num_leaves=15,
            min_child_samples=35, subsample=0.85, subsample_freq=1,
            colsample_bytree=0.75, reg_lambda=2.5, n_jobs=1, random_state=RNG_SEED, verbose=-1
        )),
    ]),
}

for m_label, m_pipe in model_zoo.items():
    run_cv_evaluation(m_label, m_pipe)

In [ ]:
model_zoo["ExtraTrees_Calibrated"] = CalibratedClassifierCV(
    model_zoo["ExtraTrees"], method="sigmoid", cv=3
)
run_cv_evaluation("ExtraTrees_Calibrated", model_zoo["ExtraTrees_Calibrated"])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6))
plot_models = ["Logistic_L2", "ExtraTrees", "ExtraTrees_Calibrated", "HistGradBoost", "LightGBM"]
plot_colors = [THEME["slate"], THEME["sand"], THEME["amber"], THEME["teal"], THEME["crimson"]]

for m_name, c_hex in zip(plot_models, plot_colors):
    f_true, f_pred = calibration_curve(y_obs, oof_predictions[m_name], n_bins=10, strategy="quantile")
    axes[0].plot(f_pred, f_true, marker="o", lw=1.8, color=c_hex, label=f"{m_name} (Brier={cv_logs[m_name].brier.mean():.4f})")
    sns.kdeplot(oof_predictions[m_name], ax=axes[1], color=c_hex, lw=1.8, label=m_name, clip=(0, 1))

axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="Ideal Diagonal")
axes[0].set_xlabel("Mean Predicted Acceptance Probability (q̂)")
axes[0].set_ylabel("Empirical Acceptance Rate")
axes[0].set_title("VI.1 Out-of-Fold Calibration Curves (10 Quantile Bins)")
axes[0].grid(True)
axes[0].legend(fontsize=8, frameon=True)

axes[1].set_xlabel("Predicted Acceptance Probability (q̂)")
axes[1].set_ylabel("Kernel Density")
axes[1].set_title("VI.2 Out-of-Fold Probability Density Profiles (KDE)")
axes[1].grid(True)
axes[1].legend(fontsize=8, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
cv_inner5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)

# Tune LightGBM
lgb_grid = {
    "m__n_estimators": [250, 350, 450, 550],
    "m__learning_rate": [0.015, 0.025, 0.035, 0.05],
    "m__num_leaves": [7, 11, 15, 23],
    "m__min_child_samples": [25, 35, 50, 70],
    "m__subsample": [0.75, 0.85, 1.0],
    "m__colsample_bytree": [0.6, 0.75, 0.9],
    "m__reg_alpha": [0.0, 0.5, 2.0],
    "m__reg_lambda": [1.0, 3.0, 8.0],
}
lgb_tuner = RandomizedSearchCV(
    model_zoo["LightGBM"], lgb_grid, n_iter=12, scoring="neg_log_loss",
    cv=cv_inner5, random_state=RNG_SEED, n_jobs=1
).fit(X_tr, y_obs)
model_zoo["LightGBM_Tuned"] = lgb_tuner.best_estimator_
print("Selected LightGBM_Tuned params:", {k.replace("m__", ""): v for k, v in lgb_tuner.best_params_.items()})
run_cv_evaluation("LightGBM_Tuned", model_zoo["LightGBM_Tuned"])

# Tune HistGradBoost
hgb_grid = {
    "m__learning_rate": [0.02, 0.03, 0.045, 0.06],
    "m__max_iter": [200, 280, 360],
    "m__max_leaf_nodes": [11, 15, 23, 31],
    "m__min_samples_leaf": [25, 40, 60, 80],
    "m__l2_regularization": [0.5, 2.0, 6.0, 12.0],
}
hgb_tuner = RandomizedSearchCV(
    model_zoo["HistGradBoost"], hgb_grid, n_iter=12, scoring="neg_log_loss",
    cv=cv_inner5, random_state=RNG_SEED, n_jobs=1
).fit(X_tr, y_obs)
model_zoo["HistGradBoost_Tuned"] = hgb_tuner.best_estimator_
print("Selected HistGradBoost_Tuned params:", {k.replace("m__", ""): v for k, v in hgb_tuner.best_params_.items()})
run_cv_evaluation("HistGradBoost_Tuned", model_zoo["HistGradBoost_Tuned"])